# Optimisation

In [1]:
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

from xgboost import XGBRegressor

In [2]:
# Loading features and the previous models
project_root = (
    Path.cwd().parent
    if Path.cwd().name == 'notebooks'
    else Path.cwd()
)

features_path = (
    project_root
    / 'data'
    / 'processed'
    / 'ca1_model_features.parquet'
)

models_directory = (
    project_root
    / 'models'
)

reports_directory = (
    project_root
    / 'reports'
)

stage8_comparison_path = (
    reports_directory
    / 'validation_model_comparison_stage8.csv'
)

feature_df = pd.read_parquet(
    features_path
)

feature_df['date'] = pd.to_datetime(
    feature_df['date']
)

feature_df = feature_df.sort_values(
    ['store_id', 'item_id', 'date']
).reset_index(drop=True)

base_rf_pipeline = joblib.load(
    models_directory
    / 'random_forest_pipeline.joblib'
)

tree_preprocessor = (
    base_rf_pipeline.named_steps[
        'preprocessor'
    ]
)

print(f'Rows: {len(feature_df):,}')
print(f'Items: {feature_df["item_id"].nunique():,}')

Rows: 166,886
Items: 108


### Reiterate the chronological split

In [3]:
all_dates = pd.DatetimeIndex(
    sorted(feature_df['date'].unique())
)

train_dates = all_dates[:-56]
validation_dates = all_dates[-56:-28]
test_dates = all_dates[-28:]

training_df = feature_df.loc[
    feature_df['date'].isin(train_dates)
].copy()

validation_df = feature_df.loc[
    feature_df['date'].isin(validation_dates)
].copy()

print(
    f'Training: {train_dates.min().date()} '
    f'to {train_dates.max().date()}'
)

print(
    f'Validation: {validation_dates.min().date()} '
    f'to {validation_dates.max().date()}'
)

print(
    f'Test held back: {test_dates.min().date()} '
    f'to {test_dates.max().date()}'
)

Training: 2011-02-26 to 2016-03-27
Validation: 2016-03-28 to 2016-04-24
Test held back: 2016-04-25 to 2016-05-22


In [4]:
# Predictors
numeric_features = [
    'lag_1',
    'lag_7',
    'lag_14',
    'lag_28',
    'rolling_mean_7',
    'rolling_mean_28',
    'rolling_std_7',
    'sell_price',
    'price_change',
    'event_indicator'
]

categorical_features = [
    'weekday',
    'month',
    'item_id',
    'dept_id',
    'cat_id'
]

model_features = (
    numeric_features
    + categorical_features
)

target = 'sales'

X_train = training_df[
    model_features
]

y_train = training_df[
    target
]

### Using the same forecasting functions

In [5]:
series_keys = [
    'store_id',
    'item_id'
]

In [6]:
def recursive_ml_forecast(
    model,
    data,
    forecast_dates,
    feature_columns,
    model_name
):
    forecast_dates = pd.DatetimeIndex(
        forecast_dates
    )

    forecast_start = forecast_dates.min()

    histories = {}

    for series_values, group in data.groupby(
        series_keys,
        sort=False
    ):
        history = (
            group.loc[
                group['date'] < forecast_start,
                'sales'
            ]
            .astype(float)
            .tolist()
        )

        if len(history) < 28:
            raise ValueError(
                f'{series_values} has fewer than '
                f'28 historical observations.'
            )

        histories[series_values] = history

    prediction_frames = []

    for forecast_date in forecast_dates:
        day_rows = data.loc[
            data['date'] == forecast_date
        ].sort_values(
            series_keys
        ).copy()

        for row_index, row in day_rows.iterrows():
            series_values = tuple(
                row[column]
                for column in series_keys
            )

            history = histories[
                series_values
            ]

            day_rows.at[
                row_index,
                'lag_1'
            ] = history[-1]

            day_rows.at[
                row_index,
                'lag_7'
            ] = history[-7]

            day_rows.at[
                row_index,
                'lag_14'
            ] = history[-14]

            day_rows.at[
                row_index,
                'lag_28'
            ] = history[-28]

            day_rows.at[
                row_index,
                'rolling_mean_7'
            ] = np.mean(history[-7:])

            day_rows.at[
                row_index,
                'rolling_mean_28'
            ] = np.mean(history[-28:])

            day_rows.at[
                row_index,
                'rolling_std_7'
            ] = np.std(
                history[-7:],
                ddof=1
            )

        assert not day_rows[
            feature_columns
        ].isna().any().any()

        raw_forecasts = model.predict(
            day_rows[feature_columns]
        )

        # Sales cannot be negative.
        clipped_forecasts = np.maximum(
            raw_forecasts,
            0
        )

        output = day_rows[
            [
                'store_id',
                'item_id',
                'dept_id',
                'cat_id',
                'date',
                'sales'
            ]
        ].copy()

        output['model'] = model_name
        output['raw_forecast'] = raw_forecasts
        output['forecast'] = clipped_forecasts

        prediction_frames.append(output)

        # Use clipped predictions to construct
        # subsequent forecast-day features.
        for row, prediction in zip(
            day_rows.itertuples(index=False),
            clipped_forecasts
        ):
            series_values = (
                row.store_id,
                row.item_id
            )

            histories[
                series_values
            ].append(float(prediction))

    return pd.concat(
        prediction_frames,
        ignore_index=True
    )

In [7]:
def calculate_prediction_metrics(
    predictions,
    prediction_column
):
    actual = predictions[
        'sales'
    ].to_numpy(dtype=float)

    forecast = predictions[
        prediction_column
    ].to_numpy(dtype=float)

    errors = actual - forecast

    mae = np.mean(
        np.abs(errors)
    )

    rmse = np.sqrt(
        np.mean(errors ** 2)
    )

    denominator = np.sum(
        np.abs(actual)
    )

    wape = (
        np.sum(np.abs(errors))
        / denominator
        if denominator > 0
        else np.nan
    )

    return {
        'MAE': mae,
        'RMSE': rmse,
        'WAPE': wape
    }

### Selected Random Forest Preliminary Hyperparameters

In [8]:
rf_candidates = [
    {
        'candidate_id': 'rf_01_baseline',
        'params': {
            'n_estimators': 100,
            'max_depth': 16,
            'min_samples_split': 2,
            'min_samples_leaf': 2,
            'max_features': 'sqrt'
        }
    },
    {
        'candidate_id': 'rf_02_more_trees',
        'params': {
            'n_estimators': 200,
            'max_depth': 16,
            'min_samples_split': 2,
            'min_samples_leaf': 2,
            'max_features': 'sqrt'
        }
    },
    {
        'candidate_id': 'rf_03_more_features',
        'params': {
            'n_estimators': 150,
            'max_depth': 16,
            'min_samples_split': 2,
            'min_samples_leaf': 2,
            'max_features': 0.5
        }
    },
    {
        'candidate_id': 'rf_04_deeper',
        'params': {
            'n_estimators': 150,
            'max_depth': 24,
            'min_samples_split': 2,
            'min_samples_leaf': 1,
            'max_features': 0.5
        }
    },
    {
        'candidate_id': 'rf_05_regularised',
        'params': {
            'n_estimators': 150,
            'max_depth': 20,
            'min_samples_split': 10,
            'min_samples_leaf': 4,
            'max_features': 0.5
        }
    },
    {
        'candidate_id': 'rf_06_all_features',
        'params': {
            'n_estimators': 150,
            'max_depth': 18,
            'min_samples_split': 5,
            'min_samples_leaf': 2,
            'max_features': 1.0
        }
    }
]

These hyperparameters are selected specifically for stability, restrictive sqrt sampling, relationship with depth and non-linear patters, and generalisation improvement.

In [9]:
# Selected XGBoost preliminary hyperparemeters
xgb_candidates = [
    {
        'candidate_id': 'xgb_01_baseline',
        'params': {
            'n_estimators': 300,
            'learning_rate': 0.05,
            'max_depth': 6,
            'subsample': 0.8,
            'colsample_bytree': 0.8
        }
    },
    {
        'candidate_id': 'xgb_02_slow_learning',
        'params': {
            'n_estimators': 500,
            'learning_rate': 0.03,
            'max_depth': 6,
            'subsample': 0.8,
            'colsample_bytree': 0.8
        }
    },
    {
        'candidate_id': 'xgb_03_shallower',
        'params': {
            'n_estimators': 400,
            'learning_rate': 0.05,
            'max_depth': 4,
            'subsample': 0.9,
            'colsample_bytree': 0.9
        }
    },
    {
        'candidate_id': 'xgb_04_deeper',
        'params': {
            'n_estimators': 400,
            'learning_rate': 0.03,
            'max_depth': 8,
            'subsample': 0.8,
            'colsample_bytree': 0.8
        }
    },
    {
        'candidate_id': 'xgb_05_faster_learning',
        'params': {
            'n_estimators': 250,
            'learning_rate': 0.08,
            'max_depth': 5,
            'subsample': 0.9,
            'colsample_bytree': 0.9
        }
    }
]

In [10]:
# Arrays for evaluation results
search_results = []

best_models = {}
best_predictions = {}
best_scores = {}

In [11]:
# evaluation functions
def evaluate_candidates(
    model_family,
    candidates
):
    for candidate in candidates:
        candidate_id = candidate[
            'candidate_id'
        ]

        params = candidate[
            'params'
        ]

        print(
            f'\nTraining {candidate_id}...'
        )

        if model_family == 'random_forest':
            estimator = RandomForestRegressor(
                **params,
                max_samples=0.8,
                n_jobs=-1,
                random_state=10
            )

        elif model_family == 'xgboost':
            estimator = XGBRegressor(
                **params,
                objective='reg:squarederror',
                eval_metric='rmse',
                tree_method='hist',
                n_jobs=-1,
                random_state=10
            )

        else:
            raise ValueError(
                f'Unknown model family: '
                f'{model_family}'
            )

        pipeline = Pipeline([
            (
                'preprocessor',
                clone(tree_preprocessor)
            ),
            (
                'regressor',
                estimator
            )
        ])

        training_start = perf_counter()

        pipeline.fit(
            X_train,
            y_train
        )

        training_seconds = (
            perf_counter()
            - training_start
        )

        forecast_start = perf_counter()

        predictions = (
            recursive_ml_forecast(
                model=pipeline,
                data=feature_df,
                forecast_dates=validation_dates,
                feature_columns=model_features,
                model_name=candidate_id
            )
        )

        forecasting_seconds = (
            perf_counter()
            - forecast_start
        )

        metrics = (
            calculate_prediction_metrics(
                predictions,
                'forecast'
            )
        )

        negative_predictions = (
            predictions[
                'raw_forecast'
            ].lt(0).sum()
        )

        result = {
            'model_family': model_family,
            'candidate_id': candidate_id,
            'training_seconds': training_seconds,
            'forecasting_seconds': forecasting_seconds,
            'negative_predictions':
                negative_predictions,
            **metrics
        }

        for parameter, value in params.items():
            result[
                f'param_{parameter}'
            ] = value

        search_results.append(result)

        candidate_score = (
            metrics['WAPE'],
            metrics['RMSE']
        )

        if (
            model_family not in best_scores
            or candidate_score
            < best_scores[model_family]
        ):
            best_scores[
                model_family
            ] = candidate_score

            best_models[
                model_family
            ] = pipeline

            best_predictions[
                model_family
            ] = predictions

        print(
            f'{candidate_id}: '
            f'WAPE={metrics["WAPE"]:.4f}, '
            f'RMSE={metrics["RMSE"]:.4f}, '
            f'training={training_seconds:.1f}s'
        )

### Fit and evaluate the results for RF

In [12]:
evaluate_candidates(
    model_family='random_forest',
    candidates=rf_candidates
)


Training rf_01_baseline...
rf_01_baseline: WAPE=0.7517, RMSE=2.2783, training=17.0s

Training rf_02_more_trees...
rf_02_more_trees: WAPE=0.7521, RMSE=2.2871, training=31.6s

Training rf_03_more_features...
rf_03_more_features: WAPE=0.7789, RMSE=2.3640, training=97.7s

Training rf_04_deeper...
rf_04_deeper: WAPE=0.8476, RMSE=2.4622, training=362.8s

Training rf_05_regularised...
rf_05_regularised: WAPE=0.7652, RMSE=2.3123, training=135.8s

Training rf_06_all_features...
rf_06_all_features: WAPE=0.7979, RMSE=2.3513, training=205.3s


In [13]:
# Result analysis
rf_search_results = (
    pd.DataFrame(search_results)
    .loc[
        lambda data:
        data['model_family']
        == 'random_forest'
    ]
    .sort_values(
        ['WAPE', 'RMSE']
    )
)

rf_search_results[
    [
        'candidate_id',
        'MAE',
        'RMSE',
        'WAPE',
        'training_seconds'
    ]
]

,candidate_id,MAE,RMSE,WAPE,training_seconds
0,rf_01_baseline,1.177787,2.278336,0.751715,17.016649
1,rf_02_more_trees,1.178461,2.287116,0.752145,31.590541
4,rf_05_regularised,1.198977,2.312318,0.765240,135.842368
2,rf_03_more_features,1.220422,2.363988,0.778927,97.714113
5,rf_06_all_features,1.250135,2.351291,0.797891,205.300533
3,rf_04_deeper,1.328084,2.462178,0.847642,362.765006


In [14]:
# Saving the results so far
reports_directory.mkdir(
    parents=True,
    exist_ok=True
)

pd.DataFrame(
    search_results
).to_csv(
    reports_directory
    / 'hyperparameter_search_checkpoint.csv',
    index=False
)

### Fit and evaluate the results for XGBoost

In [15]:
evaluate_candidates(
    model_family='xgboost',
    candidates=xgb_candidates
)


Training xgb_01_baseline...
xgb_01_baseline: WAPE=0.7961, RMSE=2.4089, training=2.8s

Training xgb_02_slow_learning...
xgb_02_slow_learning: WAPE=0.8027, RMSE=2.4097, training=4.3s

Training xgb_03_shallower...
xgb_03_shallower: WAPE=0.7991, RMSE=2.3631, training=2.9s

Training xgb_04_deeper...
xgb_04_deeper: WAPE=0.7990, RMSE=2.3909, training=4.2s

Training xgb_05_faster_learning...
xgb_05_faster_learning: WAPE=0.8065, RMSE=2.3354, training=2.2s


In [16]:
search_results_df = (
    pd.DataFrame(search_results)
    .sort_values(
        [
            'model_family',
            'WAPE',
            'RMSE'
        ]
    )
    .reset_index(drop=True)
)

search_results_df[
    [
        'model_family',
        'candidate_id',
        'MAE',
        'RMSE',
        'WAPE',
        'training_seconds',
        'forecasting_seconds',
        'negative_predictions'
    ]
]

,model_family,candidate_id,MAE,RMSE,WAPE,training_seconds,forecasting_seconds,negative_predictions
0,random_forest,rf_01_baseline,1.177787,2.278336,0.751715,17.016649,1.741478,0
1,random_forest,rf_02_more_trees,1.178461,2.287116,0.752145,31.590541,1.811737,0
2,random_forest,rf_05_regularised,1.198977,2.312318,0.765240,135.842368,1.971701,0
3,random_forest,rf_03_more_features,1.220422,2.363988,0.778927,97.714113,1.691299,0
4,random_forest,rf_06_all_features,1.250135,2.351291,0.797891,205.300533,1.761374,0
5,random_forest,rf_04_deeper,1.328084,2.462178,0.847642,362.765006,1.809016,0
6,xgboost,xgb_01_baseline,1.247358,2.408888,0.796119,2.840420,1.719436,0
7,xgboost,xgb_04_deeper,1.251905,2.390926,0.799021,4.245675,1.661186,0
8,xgboost,xgb_03_shallower,1.252022,2.363117,0.799096,2.859223,1.626759,0
9,xgboost,xgb_02_slow_learning,1.257699,2.409702,0.802719,4.312080,1.622341,0


### Getting the best results from both models

In [17]:
best_summary = (
    search_results_df
    .sort_values(
        [
            'model_family',
            'WAPE',
            'RMSE'
        ]
    )
    .drop_duplicates(
        subset='model_family',
        keep='first'
    )
    .reset_index(drop=True)
)

best_summary[
    [
        'model_family',
        'candidate_id',
        'MAE',
        'RMSE',
        'WAPE',
        'training_seconds'
    ]
]

,model_family,candidate_id,MAE,RMSE,WAPE,training_seconds
0,random_forest,rf_01_baseline,1.177787,2.278336,0.751715,17.016649
1,xgboost,xgb_01_baseline,1.247358,2.408888,0.796119,2.840420


In [18]:
# Comparing the tuned and untuned models
stage8_comparison = pd.read_csv(
    stage8_comparison_path
)

tuned_metrics = best_summary[
    [
        'model_family',
        'MAE',
        'RMSE',
        'WAPE'
    ]
].copy()

tuned_metrics['model'] = (
    tuned_metrics['model_family']
    + '_tuned'
)

tuned_metrics = tuned_metrics[
    [
        'model',
        'MAE',
        'RMSE',
        'WAPE'
    ]
]

In [19]:
complete_tuned_comparison = (
    pd.concat(
        [
            stage8_comparison[
                [
                    'model',
                    'MAE',
                    'RMSE',
                    'WAPE'
                ]
            ],
            tuned_metrics
        ],
        ignore_index=True
    )
    .drop_duplicates(
        subset='model',
        keep='last'
    )
    .sort_values(
        ['WAPE', 'RMSE']
    )
    .reset_index(drop=True)
)

complete_tuned_comparison[
    'WAPE_percentage'
] = (
    complete_tuned_comparison[
        'WAPE'
    ]
    * 100
)

complete_tuned_comparison

,model,MAE,RMSE,WAPE,WAPE_percentage
0,rolling_mean_28,1.169400,2.417961,0.746363,74.636250
1,random_forest,1.177787,2.278336,0.751715,75.171546
2,random_forest_tuned,1.177787,2.278336,0.751715,75.171546
3,xgboost,1.247358,2.408888,0.796119,79.611872
4,xgboost_tuned,1.247358,2.408888,0.796119,79.611872
5,linear_regression,1.323802,2.459765,0.844908,84.490836
6,ridge_regression,1.328547,2.462850,0.847937,84.793721
7,seasonal_naive_7,1.432540,3.151173,0.914310,91.430984
8,previous_day,1.507937,3.461905,0.962431,96.243141


### Improvement calculation from tuning

In [20]:
untuned_tree_metrics = (
    stage8_comparison.loc[
        stage8_comparison[
            'model'
        ].isin([
            'random_forest',
            'xgboost'
        ]),
        [
            'model',
            'MAE',
            'RMSE',
            'WAPE'
        ]
    ]
)

improvement_table = (
    best_summary[
        [
            'model_family',
            'candidate_id',
            'MAE',
            'RMSE',
            'WAPE'
        ]
    ]
    .merge(
        untuned_tree_metrics,
        left_on='model_family',
        right_on='model',
        suffixes=(
            '_tuned',
            '_untuned'
        )
    )
)

improvement_table[
    'WAPE_improvement'
] = (
    improvement_table[
        'WAPE_untuned'
    ]
    - improvement_table[
        'WAPE_tuned'
    ]
)

improvement_table[
    'relative_WAPE_improvement_percent'
] = (
    improvement_table[
        'WAPE_improvement'
    ]
    / improvement_table[
        'WAPE_untuned'
    ]
    * 100
)

improvement_table[
    [
        'model_family',
        'candidate_id',
        'WAPE_untuned',
        'WAPE_tuned',
        'WAPE_improvement',
        'relative_WAPE_improvement_percent'
    ]
]

,model_family,candidate_id,WAPE_untuned,WAPE_tuned,WAPE_improvement,relative_WAPE_improvement_percent
0,random_forest,rf_01_baseline,0.751715,0.751715,0.0,0.0
1,xgboost,xgb_01_baseline,0.796119,0.796119,0.0,0.0


In [24]:
best_overall_row = (
    best_summary
    .sort_values(
        ['WAPE', 'RMSE']
    )
    .iloc[0]
)

best_overall_family = (
    best_overall_row[
        'model_family'
    ]
)

best_overall_candidate = (
    best_overall_row[
        'candidate_id'
    ]
)

best_tuned_predictions = (
    best_predictions[
        best_overall_family
    ].copy()
)

print(
    f'Best model family: '
    f'{best_overall_family}'
)

print(
    f'Selected candidate: '
    f'{best_overall_candidate}'
)

print(
    f'Prediction rows: '
    f'{len(best_tuned_predictions):,}'
)

Best model family: random_forest
Selected candidate: rf_01_baseline
Prediction rows: 3,024


### Saving the current models and results

In [25]:
models_directory.mkdir(
    parents=True,
    exist_ok=True
)

reports_directory.mkdir(
    parents=True,
    exist_ok=True
)

In [27]:
for model_family, model in (
    best_models.items()
):
    model_path = (
        models_directory
        / f'{model_family}_tuned_pipeline.joblib'
    )

    joblib.dump(
        model,
        model_path,
        compress=3
    )

    print(f'Saved: {model_path}')

Saved: c:\Users\ryana\OneDrive - MMU\Documents\MSc Project\retail-demand-forecasting\models\random_forest_tuned_pipeline.joblib
Saved: c:\Users\ryana\OneDrive - MMU\Documents\MSc Project\retail-demand-forecasting\models\xgboost_tuned_pipeline.joblib


In [28]:
search_results_df.to_csv(
    reports_directory
    / 'hyperparameter_search_results.csv',
    index=False
)

best_summary.to_csv(
    reports_directory
    / 'best_hyperparameters.csv',
    index=False
)

improvement_table.to_csv(
    reports_directory
    / 'hyperparameter_improvement.csv',
    index=False
)

complete_tuned_comparison.to_csv(
    reports_directory
    / 'validation_model_comparison_tuned.csv',
    index=False
)

best_tuned_predictions.to_parquet(
    project_root
    / 'data'
    / 'processed'
    / 'ca1_validation_tuned_predictions.parquet',
    index=False
)

print('Stage 9 results saved.')

Stage 9 results saved.
